# Inter-Annotator Agreement (IAA): Reva vs Ryan

This notebook computes agreement between two annotators across two multi-label tasks:

1. **Target Label Mapping** (`reva_labels.tsv` vs `ryan_labels.tsv`)
2. **Dogwhistle Inferred Target** (`reva_glossary.tsv` vs `ryan_glossary.tsv`)

For both tasks, each item can contain 0 to N labels represented as a comma-separated string.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from tabulate import tabulate
from IPython.display import Markdown, display

In [ ]:
# Annotation TSV files live in the same directory as this notebook.
ANNOTATIONS_DIR = Path.cwd()

def parse_labels(value) -> frozenset:
    """Parse comma-separated label string to frozenset of stripped labels.
    Returns frozenset() for NaN or empty."""
    if pd.isna(value) or str(value).strip() == "":
        return frozenset()
    return frozenset(lbl.strip() for lbl in str(value).split(",") if lbl.strip())


def jaccard_similarity(a: frozenset, b: frozenset) -> float:
    if not a and not b:
        return 1.0
    union = a | b
    if not union:
        return 1.0
    return len(a & b) / len(union)


def krippendorff_alpha_jaccard(a_sets: list, b_sets: list) -> float:
    observed_disagreement = np.mean([1.0 - jaccard_similarity(a, b) for a, b in zip(a_sets, b_sets)])

    pooled = list(a_sets) + list(b_sets)
    m = len(pooled)
    if m < 2:
        return np.nan

    pairwise_sum_unordered = 0.0
    for i in range(m):
        for j in range(i + 1, m):
            pairwise_sum_unordered += 1.0 - jaccard_similarity(pooled[i], pooled[j])

    # Convert unordered-pair sum to ordered-pair mean over i != j.
    expected_disagreement = (2.0 * pairwise_sum_unordered) / (m * (m - 1))
    if np.isclose(expected_disagreement, 0.0):
        return 1.0

    return 1.0 - (observed_disagreement / expected_disagreement)


def load_task(
    reva_path: Path,
    ryan_path: Path,
    join_keys: list,
    label_col: str,
) -> pd.DataFrame:
    reva = pd.read_csv(reva_path, sep="\t", dtype=str).rename(columns={label_col: f"{label_col}_reva"})
    ryan = pd.read_csv(ryan_path, sep="\t", dtype=str).rename(columns={label_col: f"{label_col}_ryan"})

    merged = reva.merge(ryan, on=join_keys, how="inner")
    merged[f"{label_col}_reva"] = merged[f"{label_col}_reva"].apply(parse_labels)
    merged[f"{label_col}_ryan"] = merged[f"{label_col}_ryan"].apply(parse_labels)
    return merged


def compute_metrics(df: pd.DataFrame, label_col: str):
    a_sets = df[f"{label_col}_reva"].tolist()
    b_sets = df[f"{label_col}_ryan"].tolist()

    exact_match = np.mean([a == b for a, b in zip(a_sets, b_sets)])
    mean_jaccard = np.mean([jaccard_similarity(a, b) for a, b in zip(a_sets, b_sets)])

    all_labels = sorted(set().union(*a_sets).union(*b_sets))
    kappa_rows = []
    kappas = []

    for label in all_labels:
        a_vec = [1 if label in s else 0 for s in a_sets]
        b_vec = [1 if label in s else 0 for s in b_sets]

        # Skip labels that are all-zero for both annotators.
        if sum(a_vec) == 0 and sum(b_vec) == 0:
            continue

        kappa = cohen_kappa_score(a_vec, b_vec)
        if pd.isna(kappa):
            continue

        kappas.append(kappa)
        kappa_rows.append({
            "label": label,
            "kappa": float(kappa),
            "pct_reva_assigned": 100.0 * np.mean(a_vec),
            "pct_ryan_assigned": 100.0 * np.mean(b_vec),
        })

    breakdown = pd.DataFrame(kappa_rows)
    if not breakdown.empty:
        breakdown["abs_kappa"] = breakdown["kappa"].abs()
        breakdown = breakdown.sort_values("abs_kappa", ascending=False).reset_index(drop=True)

    result = {
        "n_items": len(df),
        "exact_match_pct": 100.0 * float(exact_match),
        "mean_jaccard": float(mean_jaccard),
        "macro_cohen_kappa": float(np.mean(kappas)) if kappas else np.nan,
        "krippendorff_alpha_jaccard": float(krippendorff_alpha_jaccard(a_sets, b_sets)),
        "n_labels_for_kappa": len(kappas),
    }
    return result, breakdown

In [ ]:
labels_df = load_task(
    reva_path=ANNOTATIONS_DIR / "reva_labels.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_labels.tsv",
    join_keys=["dataset", "raw_target_label"],
    label_col="dest_label",
)

glossary_df = load_task(
    reva_path=ANNOTATIONS_DIR / "reva_glossary.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_glossary.tsv",
    join_keys=["term"],
    label_col="inferred_target",
)

labels_metrics, labels_breakdown = compute_metrics(labels_df, "dest_label")
glossary_metrics, glossary_breakdown = compute_metrics(glossary_df, "inferred_target")

print(f"Labels items:   {labels_metrics['n_items']}")
print(f"Glossary items: {glossary_metrics['n_items']}")

## What each metric means (plain language)

- **Exact Match %**: how often Reva and Ryan picked exactly the same set of labels for an item.
- **Mean Jaccard Similarity**: average overlap between their label sets; partial overlap gets partial credit.
- **Macro-Averaged Cohen's κ**: for each label, treat annotation as yes/no and compute κ; then average those κ values across labels.
- **Krippendorff's α (Jaccard distance)**: agreement adjusted for chance using set-distance disagreement; implemented directly from the formula without external α packages.

In [ ]:
summary = pd.DataFrame([
    {"Metric": "N Items", "Labels Task": labels_metrics["n_items"], "Glossary Task": glossary_metrics["n_items"]},
    {"Metric": "Exact Match %", "Labels Task": f"{labels_metrics['exact_match_pct']:.2f}%", "Glossary Task": f"{glossary_metrics['exact_match_pct']:.2f}%"},
    {"Metric": "Mean Jaccard Similarity", "Labels Task": f"{labels_metrics['mean_jaccard']:.3f}", "Glossary Task": f"{glossary_metrics['mean_jaccard']:.3f}"},
    {"Metric": "Macro-Avg Cohen's κ", "Labels Task": f"{labels_metrics['macro_cohen_kappa']:.3f} (N={labels_metrics['n_labels_for_kappa']} lbl)", "Glossary Task": f"{glossary_metrics['macro_cohen_kappa']:.3f} (N={glossary_metrics['n_labels_for_kappa']} lbl)"},
    {"Metric": "Krippendorff's α (Jaccard dist)", "Labels Task": f"{labels_metrics['krippendorff_alpha_jaccard']:.3f}", "Glossary Task": f"{glossary_metrics['krippendorff_alpha_jaccard']:.3f}"},
])

def stripe_rows(row):
    color = "#f8fbff" if row.name % 2 == 0 else "#eef5fb"
    return [f"background-color: {color}"] * len(row)

styled_summary = (
    summary.style
    .set_caption("Table X: Inter-Annotator Agreement (IAA) across two annotation tasks.")
    .apply(stripe_rows, axis=1)
    .set_properties(subset=["Metric"], **{"font-weight": "bold"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#dbe9f4"), ("font-weight", "bold")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "1.05em"), ("font-weight", "bold"), ("padding", "6px")]},
    ])
    .hide(axis="index")
)

display(styled_summary)
display(Markdown(
    "κ = macro-averaged Cohen's kappa across all binary per-label vectors. "
    "α computed with Jaccard distance metric. "
    "N labels = number of distinct labels with non-trivial variance used for κ computation."
))

print(tabulate(summary, headers="keys", tablefmt="github", showindex=False))

In [ ]:
def format_kappa_breakdown(breakdown: pd.DataFrame, task_name: str, top_n: int = 20) -> pd.DataFrame:
    if breakdown.empty:
        return pd.DataFrame(columns=["label", "κ", "% Reva assigned", "% Ryan assigned"])

    top = breakdown.sort_values("abs_kappa", ascending=False).head(top_n).copy()
    out = top[["label", "kappa", "pct_reva_assigned", "pct_ryan_assigned"]].rename(columns={
        "kappa": "κ",
        "pct_reva_assigned": "% Reva assigned",
        "pct_ryan_assigned": "% Ryan assigned",
    })
    out["κ"] = out["κ"].map(lambda x: f"{x:.3f}")
    out["% Reva assigned"] = out["% Reva assigned"].map(lambda x: f"{x:.2f}%")
    out["% Ryan assigned"] = out["% Ryan assigned"].map(lambda x: f"{x:.2f}%")
    out.insert(0, "Task", task_name)
    return out

labels_top20 = format_kappa_breakdown(labels_breakdown, "Labels Task", top_n=20)
glossary_top20 = format_kappa_breakdown(glossary_breakdown, "Glossary Task", top_n=20)

display(Markdown("### Per-label κ breakdown (Top-20 by |κ|): Labels Task"))
display(labels_top20.style.hide(axis="index").set_properties(subset=["label"], **{"font-weight": "bold"}))

display(Markdown("### Per-label κ breakdown (Top-20 by |κ|): Glossary Task"))
display(glossary_top20.style.hide(axis="index").set_properties(subset=["label"], **{"font-weight": "bold"}))

---

## IAA Before vs. After Resolution

The section above reports raw Reva-vs-Ryan agreement on the original annotations.
This section evaluates how the rule-based resolution scheme (in `merge_annotation_union.ipynb`)
changed that agreement landscape.

**Three additional comparisons (per task):**

| Comparison | What it measures |
|---|---|
| **Reva vs Ryan (raw)** | Baseline annotator agreement — reproduced here with `_unknown` stripped |
| **Reva vs Resolved** | How closely Reva's labels match the adjudicated output |
| **Ryan vs Resolved** | How closely Ryan's labels match the adjudicated output |
| **Naive union vs Resolved** | How much the rule-based merge differs from the naive superset |

All four comparisons use `parse_labels_eff` (stripping `_unknown` sentinel) so that
"I don't know" is treated as an empty set rather than a positive label — consistent
with the resolution rules. The raw-IAA numbers in the table above use `parse_labels`,
so they will differ slightly from the "Reva vs Ryan" row below.

A high Jaccard for **both** annotators vs resolved means the resolution scheme stayed
close to consensus. If one annotator has systematically higher Jaccard than the other,
the resolution rules favoured that annotator's perspective.

In [ ]:
def parse_labels_eff(value) -> frozenset:
    """Like parse_labels but strips the _unknown sentinel.

    Used throughout before/after comparison so that _unknown (annotator had
    no information) is treated as an empty set, not a positive label — consistent
    with how the resolution rules operate.
    """
    return parse_labels(value) - {"_unknown"}


# ── Load resolved + naive-merged files ────────────────────────────────────────

gr = pd.read_csv(ANNOTATIONS_DIR / "merged_glossary_resolved.tsv", sep="\t", dtype=str)
lr = pd.read_csv(ANNOTATIONS_DIR / "merged_labels_resolved.tsv",   sep="\t", dtype=str)

# Attach the naive union-merged labels for before/after diff
gr_naive = (
    pd.read_csv(ANNOTATIONS_DIR / "merged_glossary.tsv", sep="\t", dtype=str)
    [["term", "inferred_target_merged"]]
    .rename(columns={"inferred_target_merged": "naive_merged"})
)
lr_naive = (
    pd.read_csv(ANNOTATIONS_DIR / "merged_labels.tsv", sep="\t", dtype=str)
    [["dataset", "raw_target_label", "dest_label_merged"]]
    .rename(columns={"dest_label_merged": "naive_merged"})
)
gr = gr.merge(gr_naive, on="term",                                     how="left")
lr = lr.merge(lr_naive, on=["dataset", "raw_target_label"],            how="left")

# Parse all label columns with _unknown stripped
for df, rv_col, ry_col, m_col in [
    (gr, "inferred_target_reva", "inferred_target_ryan", "inferred_target_merged"),
    (lr, "dest_label_reva",      "dest_label_ryan",      "dest_label_merged"),
]:
    df["reva_set"]     = df[rv_col].apply(parse_labels_eff)
    df["ryan_set"]     = df[ry_col].apply(parse_labels_eff)
    df["resolved_set"] = df[m_col].apply(parse_labels_eff)
    df["naive_set"]    = df["naive_merged"].apply(parse_labels_eff)

# ── Metric computation ─────────────────────────────────────────────────────────

def compute_pairwise(a_sets, b_sets):
    """Full IAA metric suite for two parallel lists of label frozensets."""
    exact_pct = 100.0 * float(np.mean([a == b for a, b in zip(a_sets, b_sets)]))
    mean_jacc = float(np.mean([jaccard_similarity(a, b) for a, b in zip(a_sets, b_sets)]))

    all_labels = sorted(set().union(*a_sets, *b_sets))
    kappas = []
    for lbl in all_labels:
        av = [1 if lbl in s else 0 for s in a_sets]
        bv = [1 if lbl in s else 0 for s in b_sets]
        if sum(av) == 0 and sum(bv) == 0:
            continue
        k = cohen_kappa_score(av, bv)
        if not pd.isna(k):
            kappas.append(k)

    return {
        "exact_pct":          exact_pct,
        "mean_jaccard":       mean_jacc,
        "macro_kappa":        float(np.mean(kappas)) if kappas else float("nan"),
        "n_kappa_labels":     len(kappas),
        "krippendorff_alpha": float(krippendorff_alpha_jaccard(a_sets, b_sets)),
    }


def before_after_table(df: pd.DataFrame, task_name: str) -> None:
    reva    = df["reva_set"].tolist()
    ryan    = df["ryan_set"].tolist()
    res     = df["resolved_set"].tolist()
    naive   = df["naive_set"].tolist()

    raw        = compute_pairwise(reva, ryan)
    reva_res   = compute_pairwise(reva, res)
    ryan_res   = compute_pairwise(ryan, res)
    naive_res  = compute_pairwise(naive, res)

    def _fmt(label, m, hide_kappa=False):
        kappa_str = "—" if hide_kappa else f"{m['macro_kappa']:.3f} (N={m['n_kappa_labels']})"
        alpha_str = "—" if hide_kappa else f"{m['krippendorff_alpha']:.3f}"
        return {
            "Comparison":         label,
            "Exact Match %":      f"{m['exact_pct']:.1f}%",
            "Mean Jaccard":       f"{m['mean_jaccard']:.3f}",
            "Macro Cohen's κ":    kappa_str,
            "Krippendorff's α":   alpha_str,
        }

    rows = [
        _fmt("Reva vs Ryan (raw, _unknown stripped)",   raw),
        _fmt("Reva vs Rule-based Resolved",              reva_res),
        _fmt("Ryan vs Rule-based Resolved",              ryan_res),
        _fmt("Naive union vs Rule-based Resolved",       naive_res, hide_kappa=True),
    ]
    tbl = pd.DataFrame(rows)

    # Δ columns (resolved − raw, for the two annotator rows)
    def _delta(after_str, base_str):
        try:
            return f"{float(after_str.rstrip('%')) - float(base_str.rstrip('%')):+.1f}"
        except Exception:
            return "—"

    tbl.insert(2, "Δ Exact %",   [
        "—",
        _delta(rows[1]["Exact Match %"], rows[0]["Exact Match %"]),
        _delta(rows[2]["Exact Match %"], rows[0]["Exact Match %"]),
        _delta(rows[3]["Exact Match %"], rows[0]["Exact Match %"]),
    ])
    tbl.insert(4, "Δ Jaccard",   [
        "—",
        f"{float(rows[1]['Mean Jaccard']) - float(rows[0]['Mean Jaccard']):+.3f}",
        f"{float(rows[2]['Mean Jaccard']) - float(rows[0]['Mean Jaccard']):+.3f}",
        f"{float(rows[3]['Mean Jaccard']) - float(rows[0]['Mean Jaccard']):+.3f}",
    ])

    display(Markdown(f"### {task_name}"))
    display(
        tbl.style
        .set_properties(**{"text-align": "left"})
        .set_properties(subset=["Comparison"], **{"font-weight": "bold"})
        .map(lambda v: "color: #2a6496" if isinstance(v, str) and v.startswith("+") else
                   "color: #c0392b" if isinstance(v, str) and v.startswith("-") else "",
             subset=["Δ Exact %", "Δ Jaccard"])
        .hide(axis="index")
    )
    print(tabulate(tbl, headers="keys", tablefmt="github", showindex=False))
    print()


before_after_table(gr, "Glossary Task  (n=340)")
before_after_table(lr, "Labels Task  (n=587)")

In [ ]:
# ── Resolution method coverage ────────────────────────────────────────────────

_METHOD_ORDER = [
    ("exact",                          "Exact agreement (no change)"),
    ("both_unknown",                   "Both _unknown (kept)"),
    ("rule1_unknown_vs_specific",      "Rule 1: _unknown → specific labels"),
    ("additive_superset",              "Additive superset (genuine subset, union kept)"),
    ("rule2_transgender_granularity",  "Rule 2: trans_women/unspecified → trans_unspecified"),
    ("rule3_self_referential_vs_target","Rule 3: self-referential vs target framing"),
    ("rule4_specificity_explosion",    "Rule 4: pan-group race explosion → unknown_minority"),
    ("partial_superset",               "Partial overlap (union kept)"),
    ("unresolved_disjoint",            "Unresolved disjoint (union kept, flagged ⚑)"),
    ("manual_resolution",              "Manual resolution"),
]

g_vc = gr["merge_method"].value_counts()
l_vc = lr["merge_method"].value_counts()

cov_rows = []
for key, label in _METHOD_ORDER:
    gn, ln = int(g_vc.get(key, 0)), int(l_vc.get(key, 0))
    if gn + ln == 0:
        continue
    cov_rows.append({
        "Resolution method":  label,
        "Glossary":           gn,
        "% (G)":              f"{100*gn/len(gr):.1f}%",
        "Labels":             ln,
        "% (L)":              f"{100*ln/len(lr):.1f}%",
    })

cov_df = pd.DataFrame(cov_rows)

display(Markdown("### Resolution Method Coverage"))
display(
    cov_df.style
    .set_properties(subset=["Resolution method"], **{"font-weight": "bold"})
    .map(lambda v: "background-color: #fff3cd"
              if isinstance(v, str) and "Unresolved" in v else "", subset=["Resolution method"])
    .hide(axis="index")
)
print(tabulate(cov_df, headers="keys", tablefmt="github", showindex=False))

# ── Items changed from naive union to rule-based resolution ───────────────────

display(Markdown("### Items Changed by Rule-Based Resolution"))

for df, task, merged_col in [
    (gr, "Glossary", "inferred_target_merged"),
    (lr, "Labels",   "dest_label_merged"),
]:
    changed_mask = df["resolved_set"] != df["naive_set"]
    n_changed = int(changed_mask.sum())
    n_total   = len(df)
    mean_jacc_changed = float(np.mean([
        jaccard_similarity(a, b)
        for a, b in zip(df.loc[changed_mask, "naive_set"], df.loc[changed_mask, "resolved_set"])
    ])) if n_changed > 0 else float("nan")

    print(f"{task} task: {n_changed}/{n_total} items changed "
          f"({100*n_changed/n_total:.1f}%); "
          f"mean Jaccard(naive, resolved) on changed items = {mean_jacc_changed:.3f}")

    # Per-method change count for non-trivial rules
    rule_mask = df["merge_method"].str.startswith("rule") & changed_mask
    if rule_mask.any():
        print("  Changed by rule:")
        for m, cnt in df.loc[rule_mask, "merge_method"].value_counts().items():
            print(f"    {m:<45s}: {cnt}")